# Data lookup · duplicados e identidades antes de YOLO

**Objetivo:** producir un inventario sin copias exactas, manteniendo la procedencia y las etiquetas originales.
**Pregunta:** ¿cuántas imágenes únicas tenemos y cuántas etiquetas comparten la misma imagen?
**Hipótesis comprobable:** elegir un representante por SHA-256 reduce entradas sin perder ningún hash distinto.

Fuentes: [Dogs of the World v1](https://www.kaggle.com/datasets/lextoumbourou/dogs-world) y
[LCW v1](https://www.kaggle.com/datasets/dseidli/lcwlabeled-cats-in-the-wild). La muestra es exactamente
la contenida en el CSV elegido; no se extrapolan sus cifras al 100% ni se inventan resultados.

Esta es preparación de datos: usa CPU, no llama a YOLO, CLIP, MLflow ni Hugging Face.
Exporta inventarios; no borra fotos, no fusiona identidades y no cambia las versiones usadas por E1/E2/E3.

## Fase 1. Preparación y preprocesamiento de datos

Ejecuta en una sesión **Colab CPU** con el wheel `0.2.5.dev3+lookup.378c7acc` ([instalación](README.md#data-lookup-sin-yolo)).
Monta Drive y elige `SOURCE_SAMPLE_PCT`: 20, 50 o 100. El paquete recupera el **`review.csv` raw**
del respaldo confirmado de `MyDrive/PetTrace/checkpoints/preparation`, en una carpeta local separada.

Se leen los recibos y se copia/verifica únicamente el bloque TAR que contiene ese CSV; no se restauran
las imágenes, los ZIP ni los modelos. El CSV incluye los candidatos con etiqueta identificable antes
de YOLO, no el manifiesto refinado ni necesariamente todas las fotos del ZIP.
Si ese volumen no tiene inventario respaldado, se detiene: no usa otro porcentaje ni repite la preparación.
Los resultados quedan en `MyDrive/PetTrace/data-lookup`.


In [1]:
import json
from pathlib import Path
from time import perf_counter
from IPython.display import FileLink, Markdown, display

from pettrace.data.lookup import lookup_duplicates, restore_lookup_review
from pettrace.data.review import quarantine_gallery, quarantine_groups


ModuleNotFoundError: No module named 'pettrace'

In [2]:
from google.colab import drive

WORK = Path("/content")
DRIVE_ROOT = WORK / "drive/MyDrive/PetTrace"
SOURCE_SAMPLE_PCT = 100  # Volumen que quieres auditar: 20, 50 o 100.
if not DRIVE_ROOT.parent.is_dir():
    drive.mount(str(WORK / "drive"))
OUTPUT_ROOT = DRIVE_ROOT / "data-lookup"


In [3]:
REVIEW_CSV = restore_lookup_review(DRIVE_ROOT, WORK, percent=SOURCE_SAMPLE_PCT)
print(f"Inventario raw del {SOURCE_SAMPLE_PCT}%:", REVIEW_CSV)
print("Reportes:", OUTPUT_ROOT)


FileNotFoundError: Falta /Users/alessandro/Develop/personal/pettrace-ml/datasets/public/e3-body/source-100-seed42/review.csv. Copia el review.csv raw del volumen que quieres auditar y ajusta REVIEW_CSV.

## Fase 2. Representación y entrenamiento

La representación aquí es el **SHA-256 registrado en el inventario**; no hay entrenamiento.
[SHA-256 está definido en FIPS 180-4](https://csrc.nist.gov/pubs/fips/180-4/upd1/final).
La política de limpieza es una decisión de esta investigación:

- Agrupar todas las filas por hash y escoger un representante de forma determinista.
- Conservar la relación de cada copia con su representante y todas las etiquetas de cada hash.
- Marcar las etiquetas contradictorias para revisión, sin escoger un animal como verdadero.
- No aplicar decisiones manuales previas, YOLO, mínimo de fotos ni splits en esta auditoría raw.

`duplicados sobrantes = filas del inventario − hashes distintos`.
El paquete copia el CSV de entrada y registra sus hashes y los de sus salidas; una repetición idéntica
verifica y reutiliza el reporte. No relee las imágenes: estas deberán verificarse al consumirlas.

In [ ]:
started = perf_counter()
REPORT = lookup_duplicates(REVIEW_CSV, OUTPUT_ROOT)
summary = json.loads((REPORT / "summary.json").read_text())
print(f"Inventario analizado en {perf_counter() - started:.1f} s")
print("Reporte:", REPORT)

## Fase 3. Evaluación y análisis

Métricas: archivos, hashes únicos, copias sobrantes, porcentaje duplicado, etiquetas implicadas en
conflictos y fotos distintas por etiqueta. Una etiqueta de origen no equivale a un animal validado.
Los resultados siguientes se calculan al ejecutar este notebook.

In [ ]:
print(f"Archivos: {summary['files']:,} → imágenes únicas: {summary['unique_images']:,}")
print(f"Copias exactas sobrantes: {summary['duplicate_copies']:,} ({summary['duplicate_pct']:.2f}%)")
print(f"Hashes con etiquetas contradictorias: {summary['conflicting_hashes']:,}")
print(f"Etiquetas involucradas: {summary['conflicting_identity_labels']:,}")
lines = ["| Fuente | Archivos | Hashes únicos | Copias dentro de la fuente |", "|---|---:|---:|---:|"]
for source, counts in summary["by_source"].items():
    lines.append(f"| {source} | {counts['files']:,} | {counts['unique_images']:,} | {counts['within_source_duplicate_copies']:,} |")
display(Markdown("\n".join(lines)))
print("Los conteos por fuente se calculan dentro de cada fuente; el global también cruza fuentes.")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(["Archivos", "Hashes únicos", "Copias sobrantes"],
            [summary["files"], summary["unique_images"], summary["duplicate_copies"]],
            color=["#64748b", "#2563eb", "#d97706"])
axes[0].set_title("Duplicados exactos del inventario")
axes[0].set_ylabel("Cantidad")
histogram = {int(n): count for n, count in summary["identities_by_distinct_photos"].items()}
bins = [sum(count for n, count in histogram.items() if low <= n <= high)
        for low, high in [(1, 1), (2, 2), (3, 3), (4, 9), (10, summary["files"])]]
axes[1].bar(["1", "2", "3", "4–9", "10+"], bins, color="#2563eb")
axes[1].set_title("Fotos únicas por etiqueta, sin resolver conflictos")
axes[1].set_xlabel("Fotos distintas por SHA-256")
axes[1].set_ylabel("Etiquetas de origen")
fig.tight_layout()
fig.savefig(REPORT.parent / f"{REPORT.name}-resumen.png", dpi=120)
plt.show()
plt.close(fig)

In [ ]:
groups = quarantine_groups(REPORT)
print(f"Grupos de etiquetas conectadas para revisar: {len(groups):,}")
groups_file = REPORT.parent / f"{REPORT.name}-grupos.json"
groups_file.write_text(json.dumps({"lookup_fingerprint": REPORT.name, "decision": "pending", "groups": groups}, indent=2))
display(FileLink(str(groups_file)))

### Revisión visual opcional

El conteo anterior no necesita las fotos. Esta celda solo muestra el visor si las imágenes originales
siguen disponibles en las rutas del CSV, por ejemplo en el mismo runtime de preparación. No las descarga
ni restaura. Cada imagen aparece una sola vez con sus etiquetas; las conexiones indirectas tampoco
prueban identidad. En una sesión nueva sin fotos, el visor se omite y el análisis del inventario queda completo.


In [ ]:
BATCH_START = 0  # 0, 8, 16…
if groups:
    try:
        gallery = quarantine_gallery(REPORT, REPORT.parent / f"{REPORT.name}-grupos-{BATCH_START:04d}.html",
                                     start=BATCH_START, count=8)
    except FileNotFoundError:
        print("Visor omitido: faltan fotos originales en sus rutas. El análisis del inventario está completo.")
    else:
        display(FileLink(str(gallery)))
else:
    print("No hay conflictos de etiquetas por hash en este inventario.")

### Salidas y límites

`unique-images.csv` contiene un representante por hash con **todas** sus etiquetas;
`duplicate-copies.csv` relaciona las copias con ese representante. `review.csv` conserva la entrada exacta,
`audit.json` los conflictos y `lookup.json` la procedencia e integridad del reporte.

No se eliminaron fotos ni se resolvieron identidades. El resultado **no es un dataset listo para entrenar**:
aún requiere revisión de etiquetas, controles de calidad, protección del benchmark y splits.
No detecta recomprimidos, recortes, otras fotografías del mismo animal ni errores de etiqueta sin copias exactas.
La modificación física de una foto posterior al inventario no se detecta hasta volver a verificar sus bytes.
Cambiar la limpieza para futuros experimentos requiere una versión nueva; no altera las corridas históricas.

In [ ]:
for filename in ("summary.json", "unique-images.csv", "duplicate-copies.csv", "audit.json", "lookup.json"):
    display(FileLink(str(REPORT / filename)))